<a href="https://colab.research.google.com/github/AndresMontesDeOca/NLP_1/blob/main/Desafios/Desafio_4_AndresMontesDeOca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/AndresMontesDeOca/NLP_1/blob/main/Desafios/Desafio_4._AndresMontesDeOca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">

# **Procesamiento de Lenguaje Natural**
## **Desafio 4, Traductor**

### **Consigna**

* Replicar el modelo traductor desarrollado en clase y extender su entrenamiento utilizando un conjunto de datos más amplio y secuencias de mayor longitud.
* Modificar valores de hiperparámetros y analizar su impacto en el desempeño del traductor.
* Analizar el impacto del número de neuronas en las capas recurrentes.
* Generar y presentar al menos cinco ejemplos de traducciones producidas por el modelo entrenado.
* Interpretar a detalle los resultados obtenidos, considerando métricas de evaluación, calidad de las traducciones y posibles limitaciones.

## 1. Configuración del Entorno

In [1]:
import os
os.environ["WANDB_API_KEY"] = "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"



## 2. Importación de Librerías y W&B Login

In [2]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.layers import Input, LSTM, GRU, Dense, Embedding, Dropout, Bidirectional
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
import wandb

try:
    from wandb.keras import WandbMetricsLogger as WandbLogger
except ImportError:
    try:
        from wandb.integration.keras import WandbMetricsLogger as WandbLogger
    except ImportError:
        from wandb.keras import WandbCallback as WandbLogger

# 1. Intentamos obtener la clave usando los Secrets nativos de Google Colab
wandb_key = None
try:
    from google.colab import userdata
    wandb_key = userdata.get('WANDB_API_KEY')
    print("Clave leída exitosamente desde Colab Secrets.")
except Exception:
    pass

# 2. Si no estamos en Colab o falló, hacemos un Fallback al archivo .env local
if not wandb_key:
    try:
        from dotenv import load_dotenv
        load_dotenv('.env')
        load_dotenv('Desafios/.env')
        load_dotenv('../Desafios/.env')
        wandb_key = os.environ.get("WANDB_API_KEY")
        if wandb_key:
            print("Clave leída exitosamente desde .env local.")
    except ImportError:
        pass

# 3. Hacemos el Login definitivo
if wandb_key:
    wandb.login(key=wandb_key, relogin=True)
else:
    print("ERROR FATAL: No se encontró la WANDB_API_KEY ni en Colab Secrets ni en el .env local.")


Clave leída exitosamente desde Colab Secrets.


/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: amontesdeoca1982 (andresmontesdeoca) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 3. Descarga y Preprocesamiento del Dataset

In [3]:
if not os.path.exists('spa-eng'):
    os.system("curl -L -o spa-eng.zip http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip")
    os.system("unzip -q spa-eng.zip")

with open("./spa-eng/spa.txt", encoding="utf-8") as f:
    lines = f.read().split("\n")[:-1]

MAX_NUM_SENTENCES = 30000

np.random.seed(42)
np.random.shuffle(lines)

input_sentences, output_sentences = [], []
for i, line in enumerate(lines):
    if i >= MAX_NUM_SENTENCES:
        break
    if '\t' not in line:
        continue
    input_sentence, output = line.rstrip().split('\t')[:2]
    input_sentences.append(input_sentence)
    output_sentences.append(output)


output_sentences_inputs = ['<sos> ' + s for s in output_sentences]
output_sentences_targets = [s + ' <eos>' for s in output_sentences]

MAX_VOCAB_SIZE = 10000

input_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE)
input_tokenizer.fit_on_texts(input_sentences)
input_integer_seq = input_tokenizer.texts_to_sequences(input_sentences)
word2idx_inputs = input_tokenizer.word_index

output_tokenizer = Tokenizer(
    num_words=MAX_VOCAB_SIZE,
    filters='!"#$%&()*+,-./:;=¿?@[\\]^_`{|}~\t\n'
)
output_tokenizer.fit_on_texts(output_sentences_inputs + output_sentences_targets)
output_input_integer_seq = output_tokenizer.texts_to_sequences(output_sentences_inputs)
output_target_integer_seq = output_tokenizer.texts_to_sequences(output_sentences_targets)
word2idx_outputs = output_tokenizer.word_index

max_input_len = 14
max_out_len = 16

encoder_input_sequences = pad_sequences(input_integer_seq, maxlen=max_input_len)
decoder_input_sequences = pad_sequences(output_input_integer_seq, maxlen=max_out_len, padding='post')
decoder_output_sequences = pad_sequences(output_target_integer_seq, maxlen=max_out_len, padding='post')
num_words_output = min(len(word2idx_outputs) + 1, MAX_VOCAB_SIZE)


def make_dataset(enc_seqs, dec_in_seqs, dec_out_seqs, batch_size, num_classes):
    n = len(enc_seqs)
    def generator():
        for i in range(n):
            yield (
                enc_seqs[i].astype(np.int32),
                dec_in_seqs[i].astype(np.int32),
                dec_out_seqs[i].astype(np.int32),
            )

    def encode_one_hot(enc, dec_in, dec_out):
        y = tf.one_hot(dec_out, depth=num_classes)
        return (enc, dec_in), y

    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(max_input_len,), dtype=tf.int32),
            tf.TensorSpec(shape=(max_out_len,),   dtype=tf.int32),
            tf.TensorSpec(shape=(max_out_len,),   dtype=tf.int32),
        )
    )
    ds = ds.map(encode_one_hot, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

BATCH_SIZE = 64
val_split = 0.15
split_idx = int(len(encoder_input_sequences) * (1 - val_split))

train_ds = make_dataset(
    encoder_input_sequences[:split_idx],
    decoder_input_sequences[:split_idx],
    decoder_output_sequences[:split_idx],
    BATCH_SIZE, num_words_output
)
val_ds = make_dataset(
    encoder_input_sequences[split_idx:],
    decoder_input_sequences[split_idx:],
    decoder_output_sequences[split_idx:],
    BATCH_SIZE, num_words_output
)


_PKL_PATH = 'gloveembedding.pkl'
_FILE_ID = '1KY6avD5I1eI2dxQzMkR3WExwKwRq2g94'

def _is_valid_pickle(path):
    try:
        with open(path, 'rb') as f:
            head = f.read(20)
        return b'<html' not in head.lower() and b'<!doctype' not in head.lower()
    except Exception:
        return False

if os.path.exists(_PKL_PATH) and not _is_valid_pickle(_PKL_PATH):
    os.remove(_PKL_PATH)

if not os.path.exists(_PKL_PATH):
    try:
        import gdown
        gdown.download(id=_FILE_ID, output=_PKL_PATH, quiet=True)
    except ImportError:
        os.system(f"curl -L -o {_PKL_PATH} 'https://drive.google.com/u/0/uc?id={_FILE_ID}&export=download&confirm=t'")

max_bytes = 2**28 - 1
raw = bytearray()
sz = os.path.getsize(_PKL_PATH)
with open(_PKL_PATH, 'rb') as f:
    for _ in range(0, sz, max_bytes):
        raw += f.read(max_bytes)
glove_embeddings = pickle.loads(raw)

idx_array = np.arange(glove_embeddings.shape[0])
glove_word2idx = dict(zip(glove_embeddings['word'], idx_array))

EMBED_DIM = 50
nb_words = min(MAX_VOCAB_SIZE, len(word2idx_inputs))
embedding_matrix = np.zeros((nb_words, EMBED_DIM))

for word, i in word2idx_inputs.items():
    if i < nb_words:
        idx_glove = glove_word2idx.get(word, -1)
        if idx_glove != -1:
            embedding_matrix[i] = glove_embeddings[idx_glove]['embedding']


output_sentences_inputs = ['<sos> ' + s for s in output_sentences]
output_sentences_targets = [s + ' <eos>' for s in output_sentences]

MAX_VOCAB_SIZE = 10000

input_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE)
input_tokenizer.fit_on_texts(input_sentences)
input_integer_seq = input_tokenizer.texts_to_sequences(input_sentences)
word2idx_inputs = input_tokenizer.word_index

output_tokenizer = Tokenizer(
    num_words=MAX_VOCAB_SIZE,
    filters='!"#$%&()*+,-./:;=¿?@[\\]^_`{|}~\t\n'
)
output_tokenizer.fit_on_texts(output_sentences_inputs + output_sentences_targets)
output_input_integer_seq = output_tokenizer.texts_to_sequences(output_sentences_inputs)
output_target_integer_seq = output_tokenizer.texts_to_sequences(output_sentences_targets)
word2idx_outputs = output_tokenizer.word_index

max_input_len = 14
max_out_len = 16

encoder_input_sequences = pad_sequences(input_integer_seq, maxlen=max_input_len)
decoder_input_sequences = pad_sequences(output_input_integer_seq, maxlen=max_out_len, padding='post')
decoder_output_sequences = pad_sequences(output_target_integer_seq, maxlen=max_out_len, padding='post')
num_words_output = min(len(word2idx_outputs) + 1, MAX_VOCAB_SIZE)


def make_dataset(enc_seqs, dec_in_seqs, dec_out_seqs, batch_size, num_classes):
    n = len(enc_seqs)
    def generator():
        for i in range(n):
            yield (
                enc_seqs[i].astype(np.int32),
                dec_in_seqs[i].astype(np.int32),
                dec_out_seqs[i].astype(np.int32),
            )

    def encode_one_hot(enc, dec_in, dec_out):
        y = tf.one_hot(dec_out, depth=num_classes)
        return (enc, dec_in), y

    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(max_input_len,), dtype=tf.int32),
            tf.TensorSpec(shape=(max_out_len,),   dtype=tf.int32),
            tf.TensorSpec(shape=(max_out_len,),   dtype=tf.int32),
        )
    )
    ds = ds.map(encode_one_hot, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

BATCH_SIZE = 64
val_split = 0.15
split_idx = int(len(encoder_input_sequences) * (1 - val_split))

train_ds = make_dataset(
    encoder_input_sequences[:split_idx],
    decoder_input_sequences[:split_idx],
    decoder_output_sequences[:split_idx],
    BATCH_SIZE, num_words_output
)
val_ds = make_dataset(
    encoder_input_sequences[split_idx:],
    decoder_input_sequences[split_idx:],
    decoder_output_sequences[split_idx:],
    BATCH_SIZE, num_words_output
)


_PKL_PATH = 'gloveembedding.pkl'
_FILE_ID = '1KY6avD5I1eI2dxQzMkR3WExwKwRq2g94'

def _is_valid_pickle(path):
    try:
        with open(path, 'rb') as f:
            head = f.read(20)
        return b'<html' not in head.lower() and b'<!doctype' not in head.lower()
    except Exception:
        return False

if os.path.exists(_PKL_PATH) and not _is_valid_pickle(_PKL_PATH):
    os.remove(_PKL_PATH)

if not os.path.exists(_PKL_PATH):
    try:
        import gdown
        gdown.download(id=_FILE_ID, output=_PKL_PATH, quiet=True)
    except ImportError:
        os.system(f"curl -L -o {_PKL_PATH} 'https://drive.google.com/u/0/uc?id={_FILE_ID}&export=download&confirm=t'")

max_bytes = 2**28 - 1
raw = bytearray()
sz = os.path.getsize(_PKL_PATH)
with open(_PKL_PATH, 'rb') as f:
    for _ in range(0, sz, max_bytes):
        raw += f.read(max_bytes)
glove_embeddings = pickle.loads(raw)

idx_array = np.arange(glove_embeddings.shape[0])
glove_word2idx = dict(zip(glove_embeddings['word'], idx_array))

EMBED_DIM = 50
nb_words = min(MAX_VOCAB_SIZE, len(word2idx_inputs))
embedding_matrix = np.zeros((nb_words, EMBED_DIM))

for word, i in word2idx_inputs.items():
    if i < nb_words:
        idx_glove = glove_word2idx.get(word, -1)
        if idx_glove != -1:
            embedding_matrix[i] = glove_embeddings[idx_glove]['embedding']

## 4. Carga de Embeddings Pre-entrenados (GloVe)

In [4]:
output_sentences_inputs = ['<sos> ' + s for s in output_sentences]
output_sentences_targets = [s + ' <eos>' for s in output_sentences]

MAX_VOCAB_SIZE = 10000

input_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE)
input_tokenizer.fit_on_texts(input_sentences)
input_integer_seq = input_tokenizer.texts_to_sequences(input_sentences)
word2idx_inputs = input_tokenizer.word_index

output_tokenizer = Tokenizer(
    num_words=MAX_VOCAB_SIZE,
    filters='!"#$%&()*+,-./:;=¿?@[\\]^_`{|}~\t\n'
)
output_tokenizer.fit_on_texts(output_sentences_inputs + output_sentences_targets)
output_input_integer_seq = output_tokenizer.texts_to_sequences(output_sentences_inputs)
output_target_integer_seq = output_tokenizer.texts_to_sequences(output_sentences_targets)
word2idx_outputs = output_tokenizer.word_index

max_input_len = 14
max_out_len = 16

encoder_input_sequences = pad_sequences(input_integer_seq, maxlen=max_input_len)
decoder_input_sequences = pad_sequences(output_input_integer_seq, maxlen=max_out_len, padding='post')
decoder_output_sequences = pad_sequences(output_target_integer_seq, maxlen=max_out_len, padding='post')
num_words_output = min(len(word2idx_outputs) + 1, MAX_VOCAB_SIZE)


def make_dataset(enc_seqs, dec_in_seqs, dec_out_seqs, batch_size, num_classes):
    n = len(enc_seqs)
    def generator():
        for i in range(n):
            yield (
                enc_seqs[i].astype(np.int32),
                dec_in_seqs[i].astype(np.int32),
                dec_out_seqs[i].astype(np.int32),
            )

    def encode_one_hot(enc, dec_in, dec_out):
        y = tf.one_hot(dec_out, depth=num_classes)
        return (enc, dec_in), y

    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(max_input_len,), dtype=tf.int32),
            tf.TensorSpec(shape=(max_out_len,),   dtype=tf.int32),
            tf.TensorSpec(shape=(max_out_len,),   dtype=tf.int32),
        )
    )
    ds = ds.map(encode_one_hot, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

BATCH_SIZE = 64
val_split = 0.15
split_idx = int(len(encoder_input_sequences) * (1 - val_split))

train_ds = make_dataset(
    encoder_input_sequences[:split_idx],
    decoder_input_sequences[:split_idx],
    decoder_output_sequences[:split_idx],
    BATCH_SIZE, num_words_output
)
val_ds = make_dataset(
    encoder_input_sequences[split_idx:],
    decoder_input_sequences[split_idx:],
    decoder_output_sequences[split_idx:],
    BATCH_SIZE, num_words_output
)


_PKL_PATH = 'gloveembedding.pkl'
_FILE_ID = '1KY6avD5I1eI2dxQzMkR3WExwKwRq2g94'

def _is_valid_pickle(path):
    try:
        with open(path, 'rb') as f:
            head = f.read(20)
        return b'<html' not in head.lower() and b'<!doctype' not in head.lower()
    except Exception:
        return False

if os.path.exists(_PKL_PATH) and not _is_valid_pickle(_PKL_PATH):
    os.remove(_PKL_PATH)

if not os.path.exists(_PKL_PATH):
    try:
        import gdown
        gdown.download(id=_FILE_ID, output=_PKL_PATH, quiet=True)
    except ImportError:
        os.system(f"curl -L -o {_PKL_PATH} 'https://drive.google.com/u/0/uc?id={_FILE_ID}&export=download&confirm=t'")

max_bytes = 2**28 - 1
raw = bytearray()
sz = os.path.getsize(_PKL_PATH)
with open(_PKL_PATH, 'rb') as f:
    for _ in range(0, sz, max_bytes):
        raw += f.read(max_bytes)
glove_embeddings = pickle.loads(raw)

idx_array = np.arange(glove_embeddings.shape[0])
glove_word2idx = dict(zip(glove_embeddings['word'], idx_array))

EMBED_DIM = 50
nb_words = min(MAX_VOCAB_SIZE, len(word2idx_inputs))
embedding_matrix = np.zeros((nb_words, EMBED_DIM))

for word, i in word2idx_inputs.items():
    if i < nb_words:
        idx_glove = glove_word2idx.get(word, -1)
        if idx_glove != -1:
            embedding_matrix[i] = glove_embeddings[idx_glove]['embedding']

## 5. Arquitectura del Modelo e Inferencia

In [5]:
def build_seq2seq_model(n_units=256, dropout_rate=0.3, rnn_type='lstm', bidirectional=False):
    enc_inputs = Input(shape=(max_input_len,), name='encoder_inputs')

    enc_emb_layer = Embedding(
        input_dim=nb_words,
        output_dim=EMBED_DIM,
        weights=[embedding_matrix],
        trainable=False,
        name='encoder_embedding'
    )
    enc_emb = Dropout(dropout_rate, name='encoder_dropout')(enc_emb_layer(enc_inputs))

    if rnn_type == 'gru':
        enc_rnn_layer = GRU(n_units, return_state=True, name='encoder_gru')
        if bidirectional:
            enc_rnn_layer = Bidirectional(enc_rnn_layer, name='encoder_bilstm')
            enc_out, forward_h, backward_h = enc_rnn_layer(enc_emb)
            state_h = tf.keras.layers.Concatenate()([forward_h, backward_h])
            enc_states = [state_h]
            dec_units = n_units * 2
        else:
            _, state_h = enc_rnn_layer(enc_emb)
            enc_states = [state_h]
            dec_units = n_units
    else:
        enc_rnn_layer = LSTM(n_units, return_state=True, name='encoder_lstm')
        if bidirectional:
            enc_rnn_layer = Bidirectional(enc_rnn_layer, name='encoder_bilstm')
            enc_out, forward_h, forward_c, backward_h, backward_c = enc_rnn_layer(enc_emb)
            state_h = tf.keras.layers.Concatenate()([forward_h, backward_h])
            state_c = tf.keras.layers.Concatenate()([forward_c, backward_c])
            enc_states = [state_h, state_c]
            dec_units = n_units * 2
        else:
            _, state_h, state_c = enc_rnn_layer(enc_emb)
            enc_states = [state_h, state_c]
            dec_units = n_units

    dec_inputs = Input(shape=(max_out_len,), name='decoder_inputs')

    dec_emb_layer = Embedding(
        input_dim=num_words_output,
        output_dim=dec_units,
        name='decoder_embedding'
    )
    dec_emb = Dropout(dropout_rate, name='decoder_dropout')(dec_emb_layer(dec_inputs))

    if rnn_type == 'gru':
        dec_rnn_layer = GRU(dec_units, return_sequences=True, return_state=True, name='decoder_gru')
        dec_out, _ = dec_rnn_layer(dec_emb, initial_state=enc_states)
    else:
        dec_rnn_layer = LSTM(dec_units, return_sequences=True, return_state=True, name='decoder_lstm')
        dec_out, _, _ = dec_rnn_layer(dec_emb, initial_state=enc_states)

    dec_dense_layer = Dense(num_words_output, activation='softmax', name='decoder_dense')
    dec_outputs = dec_dense_layer(dec_out)

    model = Model([enc_inputs, dec_inputs], dec_outputs)

    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        metrics=['accuracy']
    )

    return model, enc_inputs, enc_states, enc_emb_layer, enc_rnn_layer, dec_inputs, dec_emb_layer, dec_rnn_layer, dec_dense_layer


# --- Funciones de Inferencia ---
def build_encoder_inference(enc_inputs, enc_emb_layer, enc_rnn_layer, rnn_type='lstm', bidirectional=False):
    enc_emb = enc_emb_layer(enc_inputs)
    if rnn_type == 'gru':
        if bidirectional:
            enc_out, forward_h, backward_h = enc_rnn_layer(enc_emb)
            state_h = tf.keras.layers.Concatenate()([forward_h, backward_h])
            return Model(enc_inputs, [state_h])
        else:
            _, state_h = enc_rnn_layer(enc_emb)
            return Model(enc_inputs, [state_h])
    else:
        if bidirectional:
            enc_out, forward_h, forward_c, backward_h, backward_c = enc_rnn_layer(enc_emb)
            state_h = tf.keras.layers.Concatenate()([forward_h, backward_h])
            state_c = tf.keras.layers.Concatenate()([forward_c, backward_c])
            return Model(enc_inputs, [state_h, state_c])
        else:
            _, state_h, state_c = enc_rnn_layer(enc_emb)
            return Model(enc_inputs, [state_h, state_c])

def build_decoder_inference(dec_emb_layer, dec_rnn_layer, dec_dense_layer, dec_units, rnn_type='lstm'):
    dec_input_single = Input(shape=(1,), name='dec_input_single')
    dec_emb_single = dec_emb_layer(dec_input_single)

    if rnn_type == 'gru':
        dec_state_h_in = Input(shape=(dec_units,), name='dec_state_h')
        dec_out, h_out = dec_rnn_layer(dec_emb_single, initial_state=[dec_state_h_in])
        dec_out = dec_dense_layer(dec_out)
        return Model([dec_input_single, dec_state_h_in], [dec_out, h_out])
    else:
        dec_state_h_in = Input(shape=(dec_units,), name='dec_state_h')
        dec_state_c_in = Input(shape=(dec_units,), name='dec_state_c')
        dec_out, h_out, c_out = dec_rnn_layer(dec_emb_single, initial_state=[dec_state_h_in, dec_state_c_in])
        dec_out = dec_dense_layer(dec_out)
        return Model([dec_input_single, dec_state_h_in, dec_state_c_in], [dec_out, h_out, c_out])

def translate_sentence(input_seq, encoder_model, decoder_model, rnn_type='lstm'):
    states = encoder_model.predict(input_seq, verbose=0)
    if rnn_type == 'gru':
        h = states
    else:
        h, c = states

    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = word2idx_outputs['<sos>']
    eos = word2idx_outputs['<eos>']

    idx2word_target = {v: k for k, v in word2idx_outputs.items()}
    output_sentence = []

    for _ in range(max_out_len):
        if rnn_type == 'gru':
            output_tokens, h = decoder_model.predict([target_seq, h], verbose=0)
        else:
            output_tokens, h, c = decoder_model.predict([target_seq, h, c], verbose=0)

        idx = np.argmax(output_tokens[0, 0, :])
        if idx == eos:
            break
        if idx > 0:
            output_sentence.append(idx2word_target.get(idx, ''))
        target_seq[0, 0] = idx

    return ' '.join(output_sentence)

def translate(text, encoder_model, decoder_model, rnn_type='lstm'):
    seq = input_tokenizer.texts_to_sequences([text])
    seq = pad_sequences(seq, maxlen=max_input_len)
    return translate_sentence(seq, encoder_model, decoder_model, rnn_type)


def run_experiment(exp_name, n_units=256, rnn_type='lstm', bidirectional=False):
    os.environ["WANDB_SILENT"] = "true"

    wandb_key = os.environ.get("WANDB_API_KEY")
    if wandb_key:
        wandb.login(key=wandb_key, relogin=True)
    else:
        wandb.login(anonymous="allow")

    run = wandb.init(
        project="Desafio4_NLP_Traductor",
        name=exp_name,
        config={
            "learning_rate": 5e-4,
            "epochs": 30,
            "batch_size": BATCH_SIZE,
            "n_units": n_units,
            "rnn_type": rnn_type,
            "bidirectional": bidirectional,
            "dataset_size": MAX_NUM_SENTENCES
        },
        reinit=True
    )

    model, *components = build_seq2seq_model(n_units=n_units, rnn_type=rnn_type, bidirectional=bidirectional)

    lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5, verbose=1)
    early_stop = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1)

    try:
        wandb_logger = WandbLogger()
        callbacks = [lr_scheduler, early_stop, wandb_logger]
    except NameError:
        callbacks = [lr_scheduler, early_stop]

    hist = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=wandb.config.epochs,
        callbacks=callbacks,
        verbose=1
    )
    wandb.finish()

    # --- Evaluación Automática del Modelo Recién Entrenado ---
    print(f"\\n=============================================")
    print(f"  Resultados de Inferencia: {exp_name}")
    print(f"=============================================")

    enc_in, enc_states, enc_emb, enc_rnn, dec_in, dec_emb, dec_rnn, dec_dense = components
    dec_units = n_units * 2 if bidirectional else n_units

    encoder_model = build_encoder_inference(enc_in, enc_emb, enc_rnn, rnn_type=rnn_type, bidirectional=bidirectional)
    decoder_model = build_decoder_inference(dec_emb, dec_rnn, dec_dense, dec_units=dec_units, rnn_type=rnn_type)

    frases_prueba = [
        "I want to eat an apple.",
        "She is reading a very good book.",
        "What time does the train leave?",
        "We went to the beach yesterday.",
        "The weather is beautiful today, isn't it?"
    ]

    for s in frases_prueba:
        print(f"EN: {s}")
        print(f"ES: {translate(s, encoder_model, decoder_model, rnn_type=rnn_type)}\\n")

    return model, components

## 6. Experimentos de Entrenamiento

Experimento 1: LSTM (256 unidades)

In [6]:
model_1, comp_1 = run_experiment("Exp1_LSTM_256", n_units=256, rnn_type='lstm', bidirectional=False)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Epoch 1/30
    398/Unknown 41s 84ms/step - accuracy: 0.5793 - loss: 4.0370

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


399/399 ━━━━━━━━━━━━━━━━━━━━ 45s 95ms/step - accuracy: 0.6178 - loss: 2.9578 - val_accuracy: 0.6483 - val_loss: 2.4096 - learning_rate: 5.0000e-04
Epoch 2/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 34s 84ms/step - accuracy: 0.6489 - loss: 2.4015 - val_accuracy: 0.6606 - val_loss: 2.2549 - learning_rate: 5.0000e-04
Epoch 3/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 35s 87ms/step - accuracy: 0.6639 - loss: 2.2325 - val_accuracy: 0.6754 - val_loss: 2.1105 - learning_rate: 5.0000e-04
Epoch 4/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 34s 85ms/step - accuracy: 0.6758 - loss: 2.0870 - val_accuracy: 0.6861 - val_loss: 1.9934 - learning_rate: 5.0000e-04
Epoch 5/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 41s 85ms/step - accuracy: 0.6869 - loss: 1.9563 - val_accuracy: 0.6968 - val_loss: 1.8891 - learning_rate: 5.0000e-04
Epoch 6/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 41s 103ms/step - accuracy: 0.6967 - loss: 1.8432 - val_accuracy: 0.7043 - val_loss: 1.8041 - learning_rate: 5.0000e-04
Epoch 7/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 35s 87ms/step - accura

epoch/accuracy,▁▂▂▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
epoch/learning_rate,█████████████████████████████▁
epoch/loss,█▆▆▅▅▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁
epoch/val_accuracy,▁▂▃▃▄▄▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇█████████
epoch/val_loss,█▇▆▅▅▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,0.83378
epoch/epoch,29
epoch/learning_rate,0.00025
epoch/loss,0.72602
epoch/val_accuracy,0.76693


\n=============================================
  Resultados de Inferencia: Exp1_LSTM_256
EN: I want to eat an apple.
ES: quiero comprar una taza\n
EN: She is reading a very good book.
ES: ella es una buena idea de este libro\n
EN: What time does the train leave?
ES: a qué hora empieza el tren\n
EN: We went to the beach yesterday.
ES: fuimos a la playa a nadar\n
EN: The weather is beautiful today, isn't it?
ES: el verano está muy ajetreada esta noche\n


### Experimento 2: LSTM (512 unidades)

In [7]:
model_2, comp_2 = run_experiment("Exp2_LSTM_512", n_units=512, rnn_type='lstm', bidirectional=False)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Epoch 1/30
    399/Unknown 48s 116ms/step - accuracy: 0.5990 - loss: 3.4685

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


399/399 ━━━━━━━━━━━━━━━━━━━━ 54s 129ms/step - accuracy: 0.6328 - loss: 2.7244 - val_accuracy: 0.6592 - val_loss: 2.3053 - learning_rate: 5.0000e-04
Epoch 2/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 51s 128ms/step - accuracy: 0.6634 - loss: 2.2487 - val_accuracy: 0.6797 - val_loss: 2.0767 - learning_rate: 5.0000e-04
Epoch 3/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 52s 130ms/step - accuracy: 0.6820 - loss: 2.0198 - val_accuracy: 0.6959 - val_loss: 1.8976 - learning_rate: 5.0000e-04
Epoch 4/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 52s 130ms/step - accuracy: 0.6977 - loss: 1.8332 - val_accuracy: 0.7087 - val_loss: 1.7646 - learning_rate: 5.0000e-04
Epoch 5/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 82s 205ms/step - accuracy: 0.7106 - loss: 1.6779 - val_accuracy: 0.7190 - val_loss: 1.6607 - learning_rate: 5.0000e-04
Epoch 6/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 51s 127ms/step - accuracy: 0.7221 - loss: 1.5410 - val_accuracy: 0.7272 - val_loss: 1.5775 - learning_rate: 5.0000e-04
Epoch 7/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 52s 129ms/step - 

epoch/accuracy,▁▂▂▃▃▃▄▄▄▄▅▅▅▆▆▆▆▇▇▇▇█████████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
epoch/learning_rate,███████████████████▄▄▄▂▂▂▁▁▁▁▁
epoch/loss,█▇▆▅▅▄▄▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▂▃▄▄▅▅▆▆▆▇▇▇▇▇▇▇▇████████████
epoch/val_loss,█▆▅▄▄▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,0.90349
epoch/epoch,29
epoch/learning_rate,3e-05
epoch/loss,0.41381
epoch/val_accuracy,0.78025


\n=============================================
  Resultados de Inferencia: Exp2_LSTM_512
EN: I want to eat an apple.
ES: quiero comprar un\n
EN: She is reading a very good book.
ES: ella está escribiendo una buena historia\n
EN: What time does the train leave?
ES: a qué hora viene el tren\n
EN: We went to the beach yesterday.
ES: ayer fuimos a la playa\n
EN: The weather is beautiful today, isn't it?
ES: el clima es muy bien esta noche\n


### Experimento 3: GRU (256 unidades)

In [8]:
model_3, comp_3 = run_experiment("Exp3_GRU_256", n_units=256, rnn_type='gru', bidirectional=False)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Epoch 1/30
    398/Unknown 32s 76ms/step - accuracy: 0.5844 - loss: 4.0703

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


399/399 ━━━━━━━━━━━━━━━━━━━━ 37s 88ms/step - accuracy: 0.6209 - loss: 2.9651 - val_accuracy: 0.6494 - val_loss: 2.3740 - learning_rate: 5.0000e-04
Epoch 2/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 41s 87ms/step - accuracy: 0.6528 - loss: 2.3238 - val_accuracy: 0.6688 - val_loss: 2.1523 - learning_rate: 5.0000e-04
Epoch 3/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 34s 84ms/step - accuracy: 0.6742 - loss: 2.0936 - val_accuracy: 0.6867 - val_loss: 1.9688 - learning_rate: 5.0000e-04
Epoch 4/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 34s 85ms/step - accuracy: 0.6889 - loss: 1.9105 - val_accuracy: 0.6996 - val_loss: 1.8353 - learning_rate: 5.0000e-04
Epoch 5/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 41s 84ms/step - accuracy: 0.7019 - loss: 1.7618 - val_accuracy: 0.7088 - val_loss: 1.7336 - learning_rate: 5.0000e-04
Epoch 6/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 34s 86ms/step - accuracy: 0.7121 - loss: 1.6367 - val_accuracy: 0.7171 - val_loss: 1.6507 - learning_rate: 5.0000e-04
Epoch 7/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 34s 85ms/step - accurac

epoch/accuracy,▁▂▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇█████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
epoch/learning_rate,███████████████████████████▁▁▁
epoch/loss,█▆▅▅▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▂▃▄▄▅▅▆▆▆▆▇▇▇▇▇▇▇████████████
epoch/val_loss,█▇▅▅▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,0.86348
epoch/epoch,29
epoch/learning_rate,0.00025
epoch/loss,0.59093
epoch/val_accuracy,0.77271


\n=============================================
  Resultados de Inferencia: Exp3_GRU_256
EN: I want to eat an apple.
ES: quiero comprar una\n
EN: She is reading a very good book.
ES: ella es una buena libro\n
EN: What time does the train leave?
ES: a qué hora empieza el tren\n
EN: We went to the beach yesterday.
ES: fuimos a la playa a nado\n
EN: The weather is beautiful today, isn't it?
ES: el verano es el mejor clima\n


### Experimento 4: Bidirectional LSTM (256 unidades)

In [9]:
model_4, comp_4 = run_experiment("Exp4_BiLSTM_256", n_units=256, rnn_type='lstm', bidirectional=True)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Epoch 1/30
    399/Unknown 51s 121ms/step - accuracy: 0.5992 - loss: 3.4818

/usr/local/lib/python3.13/dist-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


399/399 ━━━━━━━━━━━━━━━━━━━━ 56s 135ms/step - accuracy: 0.6331 - loss: 2.7217 - val_accuracy: 0.6593 - val_loss: 2.2991 - learning_rate: 5.0000e-04
Epoch 2/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 82s 205ms/step - accuracy: 0.6634 - loss: 2.2389 - val_accuracy: 0.6799 - val_loss: 2.0666 - learning_rate: 5.0000e-04
Epoch 3/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 53s 134ms/step - accuracy: 0.6815 - loss: 2.0120 - val_accuracy: 0.6946 - val_loss: 1.8943 - learning_rate: 5.0000e-04
Epoch 4/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 53s 133ms/step - accuracy: 0.6962 - loss: 1.8287 - val_accuracy: 0.7057 - val_loss: 1.7650 - learning_rate: 5.0000e-04
Epoch 5/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 53s 133ms/step - accuracy: 0.7083 - loss: 1.6774 - val_accuracy: 0.7155 - val_loss: 1.6673 - learning_rate: 5.0000e-04
Epoch 6/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 53s 133ms/step - accuracy: 0.7192 - loss: 1.5473 - val_accuracy: 0.7233 - val_loss: 1.5879 - learning_rate: 5.0000e-04
Epoch 7/30
399/399 ━━━━━━━━━━━━━━━━━━━━ 53s 132ms/step - 

epoch/accuracy,▁▂▂▃▃▃▄▄▄▄▅▅▅▆▆▆▇▇▇▇▇█████
epoch/epoch,▁▁▂▂▂▂▃▃▃▄▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
epoch/learning_rate,███████████████████▄▄▄▄▂▂▁
epoch/loss,█▆▆▅▅▄▄▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁
epoch/val_accuracy,▁▂▃▄▄▅▅▆▆▆▇▇▇▇▇▇▇█████████
epoch/val_loss,█▆▅▄▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,0.88412
epoch/epoch,25
epoch/learning_rate,6e-05
epoch/loss,0.50017
epoch/val_accuracy,0.77435


\n=============================================
  Resultados de Inferencia: Exp4_BiLSTM_256
EN: I want to eat an apple.
ES: quiero comprar una manzana\n
EN: She is reading a very good book.
ES: ella es una buena novela\n
EN: What time does the train leave?
ES: a qué hora llega el tren\n
EN: We went to the beach yesterday.
ES: fuimos a londres ayer\n
EN: The weather is beautiful today, isn't it?
ES: el verano es la mejor derecha y la lluvia\n


### **Análisis de Resultados y Conclusiones**

#### 1. Impacto del número de neuronas y tipos de RNN
* **LSTM 256 vs 512:** Aumentar la capacidad del modelo de 256 a 512 neuronas mejoró el rendimiento general. El modelo de 512 logró una mejor `val_accuracy` (0.780 vs 0.767) y un menor `val_loss` (1.272 vs 1.324), además de un accuracy en entrenamiento notablemente mayor (0.903). En cuanto a traducciones, el de 512 generó frases mucho más coherentes (ej. *"ayer fuimos a la playa"* vs *"fuimos a la playa a nadar"*).
* **LSTM vs GRU:** El modelo GRU (256 unidades) presentó resultados intermedios (`val_accuracy` de 0.772), superando levemente a la LSTM simple de 256 y convergiendo con métricas similares pero con un menor costo computacional (menos compuertas internas). Sin embargo, en algunas oraciones la generación se vio un poco cortada o con errores gramaticales leves.
* **Bidirectional LSTM:** Al procesar las secuencias en ambas direcciones, el modelo BiLSTM de 256 unidades logró un desempeño superior al LSTM estándar de 256 (`val_accuracy` de 0.774 y `accuracy` de 0.884). Logró captar mejor el sentido de algunas palabras (ej. *"apple"* -> *"manzana"*), aunque introdujo algunas "alucinaciones" inesperadas al perder el contexto (ej. tradujo *"beach"* como *"londres"*).

#### 2. Calidad de las Traducciones
* **Aciertos:** Los modelos logran aprender correctamente la sintaxis básica y mapear sustantivos o conceptos clave en oraciones cortas (ej. "train" -> "tren", "beach" -> "playa"). El modelo LSTM-512 demostró ser el más robusto al traducir de forma perfecta *"We went to the beach yesterday"* a *"ayer fuimos a la playa"*.
* **Limitaciones:** Existe una fuerte tendencia a memorizar o confundir verbos frecuentes (ej. casi todos traducen *"eat"* como *"comprar"*). En oraciones largas o complejas (como la pregunta sobre el clima), los modelos colapsan y generan texto sin sentido (*"el verano es la mejor derecha y la lluvia"*). Esto evidencia el problema del "cuello de botella" (bottleneck) del vector de contexto fijo en arquitecturas Seq2Seq simples sin mecanismos de atención.

#### 3. Conclusión sobre el Escalamiento de Datos
* Replicar el modelo con secuencias de mayor longitud y más datos resalta la necesidad de arquitecturas más complejas. Si bien incrementar las neuronas (512) o usar redes bidireccionales ayuda a retener más información y mejora las métricas, la degradación en oraciones largas nos indica que **el siguiente paso necesario es la implementación de Mecanismos de Atención (Attention)** o la transición a modelos basados en **Transformers**, los cuales no dependen de un solo vector de contexto final para generar la traducción completa.